# 🐍 Curso de Python Básico: Aula 13 - Introdução ao Pandas

**Bem-vindo à décima terceira aula!** Como seu professor de programação do CIAA-LPS, elaborei este notebook para apresentar a principal ferramenta de manipulação e análise de dados estruturados em Python: a biblioteca **Pandas**.

Na gestão de um laboratório de pesquisa de submarinos como o CIAA-LPS, coletamos grandes quantidades de dados tabulares (arquivos CSV, planilhas Excel, tabelas SQL) com cadastros de tripulação, logs de manutenção física das embarcações e telemetria de sensores. O Pandas nos permite carregar, limpar, filtrar, agrupar e analisar esses dados com facilidade de forma tabulada (em linhas e colunas).

---

### 🎯 Objetivos da Aula
1. **Conhecer o Pandas**: Entender o propósito da biblioteca e suas duas estruturas fundamentais: `Series` e `DataFrame`.
2. **Criar Estruturas**: Aprender a inicializar Series e DataFrames a partir de dicionários e listas.
3. **Importar Dados (I/O)**: Carregar dados externos a partir de arquivos CSV.
4. **Inspecionar Dados**: Usar funções analíticas básicas como `.head()`, `.info()`, `.describe()` e `.shape`.
5. **Filtragem e Seleção**: Acessar colunas, usar `.loc[]` / `.iloc[]` e aplicar filtros com máscaras booleanas.
6. **Manipulação de Dados**: Criar novas colunas e tratar dados faltantes (`NaN`) de forma eficiente.
7. **Agrupamento (`groupby`)**: Sumarizar e agregar informações categóricas do dataset.
8. **Exercícios Práticos**: Manipular registros de telemetria crítica e históricos de manutenção.


## 1. O que é o Pandas?

O **Pandas** é uma biblioteca de código aberto construída sobre o NumPy que fornece ferramentas de análise de dados e estruturas de dados de alta performance e fáceis de usar.

### Estruturas de Dados Fundamentais:
* **Series**: É um array unidimensional rotulado (como uma única coluna de uma planilha ou uma lista com índices nomeados).
* **DataFrame**: É uma estrutura de dados bidimensional, tabular e mutável (como uma tabela de banco de dados ou uma planilha inteira do Excel). Cada coluna de um DataFrame é uma `Series`.

### Convenção de Importação
Por convenção, importamos o pandas com a abreviação `pd`:


In [ ]:
import pandas as pd
import numpy as np

# Verificando a versão instalada no contêiner
print("Versão do Pandas:", pd.__version__)


## 2. Criando Series e DataFrames

Podemos instanciar essas estruturas manualmente para modelar pequenos volumes de dados.


In [ ]:
# 1. Criando uma Series (Dados unidimensionais)
# O Pandas atribui índices (índices padrão começam em 0)
pressao_sensor = pd.Series([101.3, 101.5, 99.8, 102.1])
print("Exemplo de Series:\n", pressao_sensor)

# 2. Criando um DataFrame a partir de um dicionário (Dados bidimensionais)
# As chaves do dicionário tornam-se os cabeçalhos das colunas
dados_tripulacao = {
    "Nome": ["Carlos", "Ana", "Roberto", "Marina"],
    "Patente": ["Cabo", "Sargento", "Tenente", "Cabo"],
    "Idade": [24, 29, 35, 27],
    "Horas_Mergulho": [150, 480, 1200, 310]
}

df_tripulantes = pd.DataFrame(dados_tripulacao)
print("\nExemplo de DataFrame:\n", df_tripulantes)


## 3. Leitura e Escrita de Arquivos (I/O)

A forma mais comum de trabalhar com Pandas é lendo arquivos armazenados no disco. A biblioteca suporta uma variedade de formatos: CSV (`pd.read_csv`), Excel (`pd.read_excel`), JSON (`pd.read_json`), parquet, etc.

Vamos simular a criação de um arquivo CSV de telemetria de sensores no disco e carregá-lo usando o Pandas:


In [ ]:
# Criando um arquivo CSV simulado para testes
conteudo_csv = """timestamp,sensor_id,temperatura,status
2026-07-16 10:00:00,SENSOR-01,245.5,Normal
2026-07-16 10:01:00,SENSOR-01,248.2,Normal
2026-07-16 10:02:00,SENSOR-01,310.4,Crítico
2026-07-16 10:03:00,SENSOR-02,150.1,Normal
2026-07-16 10:04:00,SENSOR-02,,Normal
2026-07-16 10:05:00,SENSOR-01,244.9,Normal
"""

caminho_csv = "/tmp/telemetria_sensores.csv"
with open(caminho_csv, "w", encoding="utf-8") as f:
    f.write(conteudo_csv.strip())

# Carregando o arquivo CSV usando o Pandas
df_sensores = pd.read_csv(caminho_csv)
print("Arquivo carregado com sucesso! Primeiras linhas:\n", df_sensores.head(3))


## 4. Inspecionando e Explorando um DataFrame

Quando carregamos um conjunto de dados pela primeira vez, precisamos entender seu formato, colunas e tipos de dados utilizando funções utilitárias:


In [ ]:
# 1. Visualizar as primeiras ou últimas linhas
print("--- Primeiras Linhas (.head) ---")
print(df_sensores.head(2))

# 2. Informações gerais sobre tipos de dados e memória
print("\n--- Informações Gerais (.info) ---")
df_sensores.info()

# 3. Estatísticas descritivas para colunas numéricas
print("\n--- Estatísticas Descritivas (.describe) ---")
print(df_sensores.describe())

# 4. Propriedade shape (Formato: Linhas x Colunas)
print(f"\nDimensões do DataFrame (linhas, colunas): {df_sensores.shape}")


## 5. Seleção e Filtragem de Dados

### Acessando Colunas:
* Coluna única: `df['coluna']` (retorna uma Series).
* Múltiplas colunas: `df[['coluna_a', 'coluna_b']]` (retorna um DataFrame).

### Acessando Linhas:
* `.loc[rotulo]`: Acessa linhas por rótulos (nomes dos índices).
* `.iloc[posicao]`: Acessa linhas por posições numéricas inteiras (índices baseados em 0).


In [ ]:
# Acessando apenas uma coluna
print("Nomes dos Tripulantes:")
print(df_tripulantes["Nome"])

# Acessando múltiplas colunas
print("\nDados Específicos:")
print(df_tripulantes[["Nome", "Horas_Mergulho"]])

# Acessando a primeira linha pelo índice numérico (.iloc)
print("\nPrimeira linha inteira:")
print(df_tripulantes.iloc[0])


### Filtragem Condicional

Assim como no NumPy, podemos filtrar linhas do DataFrame usando operadores de comparação. Isso é útil para isolar eventos críticos do sistema.


In [ ]:
# Filtrar tripulantes experientes (Horas de Mergulho maior ou igual a 400)
filtro_experiencia = df_tripulantes["Horas_Mergulho"] >= 400
print("Tripulantes Experientes:\n", df_tripulantes[filtro_experiencia])

# Filtrar apenas sensores em estado 'Crítico' do nosso arquivo CSV
sensores_criticos = df_sensores[df_sensores["status"] == "Crítico"]
print("\nSensores em Estado Crítico:\n", sensores_criticos)


## 6. Manipulação de Colunas e Dados Faltantes

### Adicionando Novas Colunas:
Podemos computar novos atributos associados diretamente às colunas existentes.


In [ ]:
# Adicionar coluna indicando se o tripulante necessita de reciclagem (horas de mergulho menores que 350)
df_tripulantes["Reciclagem_Necessaria"] = df_tripulantes["Horas_Mergulho"] < 350
print(df_tripulantes)


### Tratamento de Dados Faltantes (`NaN`):

Em conjuntos de dados reais, é comum haver sensores desligados ou falhas que geram dados nulos (`NaN` - Not a Number). O Pandas possui funções para tratar isso:
* `.isnull()`: Retorna um DataFrame booleano indicando onde há dados faltantes.
* `.fillna(valor)`: Substitui todos os valores faltantes pelo valor especificado.
* `.dropna()`: Remove todas as linhas que contenham pelo menos um valor faltante.


In [ ]:
print("Visualizando dados nulos:")
print(df_sensores.isnull())

# Preencher temperatura faltante com a média das temperaturas
media_temperatura = df_sensores["temperatura"].mean()
df_sensores_tratado = df_sensores.copy()
df_sensores_tratado["temperatura"] = df_sensores_tratado["temperatura"].fillna(media_temperatura)

print(f"\nTemperatura média utilizada para preenchimento: {media_temperatura:.1f}")
print("\nDataFrame após tratamento (.fillna):\n", df_sensores_tratado)


## 7. Agrupamentos e Agregações (Group By)

O método `.groupby()` permite dividir o DataFrame em grupos com base em uma coluna categórica (como patente ou ID do sensor) e aplicar funções agregadas (como média, soma, contagem) para sumarizar as informações.


In [ ]:
# Agrupar a tripulação por Patente e calcular a média de Idade e Horas de Mergulho
# O agrupamento nos ajuda a ver estatísticas agregadas de cada subgrupo
resumo_patentes = df_tripulantes.groupby("Patente")[["Idade", "Horas_Mergulho"]].mean()
print("Média por Patente:\n", resumo_patentes)

# Contar quantos registros de sensores existem para cada sensor_id
contagem_sensores = df_sensores.groupby("sensor_id")["timestamp"].count()
print("\nContagem de leituras por sensor:\n", contagem_sensores)


## 8. Exercícios Práticos

---
### Exercício 1: Filtragem de Telemetria Crítica de Oxigênio

Você recebeu um DataFrame de telemetria de sensores de oxigênio de um submarino. Níveis abaixo de **19.5%** representam perigo iminente para a tripulação.

```python
dados_oxigenio = {
    "compartimento": ["Ponte", "Sala de Máquinas", "Alojamento", "Ponte", "Sala de Máquinas"],
    "oxigenio_pct": [20.9, 19.1, 20.8, 19.4, 20.9],
    "pressao_atm": [1.0, 1.1, 1.0, 0.9, 1.0]
}
```

1. Crie um DataFrame chamado `df_oxigenio` a partir do dicionário `dados_oxigenio`.
2. Filtre e exiba apenas as linhas que indicam níveis de oxigênio críticos (menor que 19.5).
3. Exiba apenas os nomes dos compartimentos que estão com oxigênio crítico.


In [ ]:
# Escreva aqui sua resolução para o Exercício 1
dados_oxigenio = {
    "compartimento": ["Ponte", "Sala de Máquinas", "Alojamento", "Ponte", "Sala de Máquinas"],
    "oxigenio_pct": [20.9, 19.1, 20.8, 19.4, 20.9],
    "pressao_atm": [1.0, 1.1, 1.0, 0.9, 1.0]
}

# 1. Criar o DataFrame...


# 2. Filtrar...


# 3. Exibir compartimentos críticos...


---
### Exercício 2: Histórico de Manutenção de Submarinos

Durante a manutenção anual, registramos os custos e os tempos de reparo dos submarinos do laboratório CIAA-LPS:

```python
dados_manutencao = {
    "submarino": ["CIAA-Alpha", "CIAA-Beta", "CIAA-Alpha", "CIAA-Beta", "CIAA-Alpha"],
    "tipo_reparo": ["Sonar", "Motores", "Oxigênio", "Casco", "Sonar"],
    "custo_usd": [15000.0, 45000.0, 8000.0, 60000.0, 12000.0],
    "dias_reparo": [3, 10, 2, 14, 4]
}
```

1. Crie um DataFrame chamado `df_manutencao` a partir do dicionário `dados_manutencao`.
2. Calcule o custo total de manutenção gasto em cada submarino (dica: agrupe por `submarino` e some os custos com `.sum()`).
3. Calcule o tempo médio (em dias) que cada submarino fica parado para manutenção.


In [ ]:
# Escreva aqui sua resolução para o Exercício 2
dados_manutencao = {
    "submarino": ["CIAA-Alpha", "CIAA-Beta", "CIAA-Alpha", "CIAA-Beta", "CIAA-Alpha"],
    "tipo_reparo": ["Sonar", "Motores", "Oxigênio", "Casco", "Sonar"],
    "custo_usd": [15000.0, 45000.0, 8000.0, 60000.0, 12000.0],
    "dias_reparo": [3, 10, 2, 14, 4]
}

# 1. Criar o DataFrame...


# 2. Custo total por submarino...


# 3. Tempo médio parado por submarino...


---

### 🎉 Parabéns!
Você concluiu a **Aula 13**! Agora você possui o conhecimento prático elementar para carregar, organizar e realizar cálculos estatísticos em tabelas de dados usando o **Pandas**, preparando-se para projetos complexos de visualização de dados e Machine Learning.
